In [1]:
import gymnasium
import numpy as np
import torch
import yaml
from PIL import Image
from torchvision.transforms import v2
from einops import rearrange

from metaworld_env import setup_metaworld_env

from model.encoder import FrameObservationEncoderNet, EncoderNet
from model.actor import DDPGActorNet
from model.critic import CriticNet

In [2]:
task_name = "push"
seed = 0

In [3]:
def setup_env(task_name:str, seed:int, render_mode:str|None = None) -> gymnasium.Env:
    env = setup_metaworld_env(task_name, False, seed, render_mode)
    env = gymnasium.wrappers.RecordEpisodeStatistics(env)
    env = gymnasium.wrappers.AddRenderObservation(env, render_only=True)
    env = gymnasium.wrappers.FrameStackObservation(env, 2)

    return env

In [4]:
with open("configs/ddpg.yaml", "r") as file:
    config = yaml.safe_load(file)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
envs = setup_env(task_name, 5000, "rgb_array")

obs_dim = envs.observation_space.shape
action_dim = envs.action_space.shape

In [5]:
encoder = EncoderNet(39, config["encoder_layers"]).to(device)
actor = DDPGActorNet(encoder.dim, np.prod(action_dim), config["actor_layers"]).to(device)
critic = CriticNet(encoder.dim, np.prod(action_dim), config["critic_layers"]).to(device)

encoder_weight, actor_weight, critic_weight = torch.load(f"weights/ddpg/{task_name}/actor_{seed}_100.pt", weights_only=True)
#encoder.load_state_dict(encoder_weight)
actor.load_state_dict(actor_weight)
critic.load_state_dict(critic_weight)

frame_observation_encoder = FrameObservationEncoderNet(6, 256).to(device)
frame_observation_encoder_weight = torch.load(f"weights/ddpg/{task_name}/frame_encoder_{seed}_mse.pt", weights_only=True)
frame_observation_encoder.load_state_dict(frame_observation_encoder_weight)

encoder = frame_observation_encoder

encoder = encoder.eval()
actor = actor.eval()

transform = v2.Compose([
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize((0.485, 0.456, 0.406, 0.485, 0.456, 0.406), (0.229, 0.224, 0.225, 0.229, 0.224, 0.225)),
        ])

In [6]:
def preprpcess(obs_batch):
    obs_batch = rearrange(obs_batch, "b l h w c -> b (l c) h w")
    obs_batch = transform(obs_batch)
    return obs_batch

@torch.no_grad()
def get_action(obs_batch, deterministic, random):

    #obs_batch = torch.as_tensor(obs_batch, dtype=torch.float32).to(device)
    obs_batch = torch.as_tensor(obs_batch).unsqueeze(0).to(device)
    obs_batch = preprpcess(obs_batch)
    obs_batch = encoder(obs_batch)
    dist = actor(obs_batch, 1)
    if deterministic:
        action = dist.mean
    else:    
        action = dist.sample(clip=None)

        if random:
            action.uniform_(-1, 1)
    
    action = action.cpu().numpy()
    
    return action.tolist()

In [7]:
success_buffer = []
for _ in range(100):
    success = 0
    obs, info = envs.reset()

    for i in range(500):
        action = get_action(obs, True, False)
        next_obs, reward, done, timeout, info = envs.step(action[0])
        success += info["success"]
        obs = next_obs
    
    success_buffer.append(success/500)

In [8]:
np.mean(success_buffer)

np.float64(0.39251999999999987)

In [9]:
success_buffer

[0.086,
 0.274,
 0.624,
 0.908,
 0.53,
 0.092,
 0.158,
 0.84,
 0.65,
 0.878,
 0.462,
 0.788,
 0.666,
 0.28,
 0.664,
 0.054,
 0.09,
 0.192,
 0.238,
 0.17,
 0.9,
 0.196,
 0.07,
 0.078,
 0.036,
 0.124,
 0.432,
 0.356,
 0.19,
 0.044,
 0.106,
 0.014,
 0.188,
 0.102,
 0.054,
 0.034,
 0.922,
 0.012,
 0.292,
 0.874,
 0.254,
 0.418,
 0.88,
 0.128,
 0.094,
 0.21,
 0.074,
 0.102,
 0.862,
 0.912,
 0.202,
 0.66,
 0.864,
 0.196,
 0.186,
 0.186,
 0.814,
 0.152,
 0.248,
 0.558,
 0.91,
 0.332,
 0.578,
 0.22,
 0.172,
 0.858,
 0.172,
 0.248,
 0.55,
 0.714,
 0.248,
 0.086,
 0.68,
 0.088,
 0.816,
 0.664,
 0.038,
 0.122,
 0.272,
 0.892,
 0.912,
 0.922,
 0.696,
 0.104,
 0.854,
 0.336,
 0.312,
 0.134,
 0.48,
 0.126,
 0.072,
 0.924,
 0.234,
 0.1,
 0.84,
 0.628,
 0.126,
 0.922,
 0.102,
 0.1]